In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("TRAIN columns:", train_df.columns.tolist())
print(train_df.head(2))
print("\nTEST columns:", test_df.columns.tolist())
print(test_df.head(2))

TRAIN columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   C  \
0  Martin Heidegger does not believe in the exist...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   D  \
0  Martin Heidegger believes that the relationshi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   E answer  
0  Martin Heid

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================================
# Setup
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: point these at your fine-tuned checkpoint directories/paths
DEBERTA_PATH = "path/to/finetuned-deberta-v3-small"
ROBERTA_PATH = "path/to/finetuned-roberta-base"

deberta_tok = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH).to(device).eval()

roberta_tok = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH).to(device).eval()

LABELS = ["A", "B", "C", "D", "E"]

test_df = pd.read_csv("test.csv")

# TODO: adjust to your actual prompt column name
PROMPT_COL = "prompt"
ID_COL = "id"

# =========================================================
# Helper: get softmax probs from a model for a given prompt
# =========================================================
def get_probs(model, tokenizer, text, max_length=512):
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=max_length, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    return probs

def top3_string(probs):
    order = np.argsort(-probs)[:3]
    return " ".join(LABELS[i] for i in order)

def top1_label(probs):
    return LABELS[int(np.argmax(probs))]

# =========================================================
# Q1 & Q2 & Q3 & Q4 — row index 25
# =========================================================
row25_prompt = test_df.loc[25, PROMPT_COL]

p_deberta_25 = get_probs(deberta_model, deberta_tok, row25_prompt)
p_roberta_25 = get_probs(roberta_model, roberta_tok, row25_prompt)

# Q1
q1_label = top1_label(p_deberta_25)
q1_prob = round(float(np.max(p_deberta_25)), 4)
print(f"Q1: {q1_label}, probability of {q1_label} = {q1_prob}")

# Q2 — simple average
p_avg_25 = (p_deberta_25 + p_roberta_25) / 2
q2_label = top1_label(p_avg_25)
print(f"Q2: {q2_label}")

# Q3 — weighted ensemble
p_weighted_25 = 0.7 * p_deberta_25 + 0.3 * p_roberta_25
q3_label = top1_label(p_weighted_25)
print(f"Q3: {q3_label}")

# Q4 — top-3 string from weighted ensemble
q4_top3 = top3_string(p_weighted_25)
print(f"Q4: {q4_top3}")

# =========================================================
# Q5 — full weighted ensemble pipeline on test.csv -> submission.csv
# =========================================================
submission_rows = []
for idx, row in test_df.iterrows():
    prompt = row[PROMPT_COL]
    p_d = get_probs(deberta_model, deberta_tok, prompt)
    p_r = get_probs(roberta_model, roberta_tok, prompt)
    p_w = 0.7 * p_d + 0.3 * p_r
    submission_rows.append({
        "id": row[ID_COL],
        "prediction": top3_string(p_w)
    })

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)

q5_answer = len(submission_df)
print(f"Q5: {q5_answer}")

# =========================================================
# Q6 — TTA on first 50 rows (DeBERTa only)
# =========================================================
INSTRUCTION_PREFIX = "Answer the following multiple-choice question carefully: "

tta_diff_count = 0
for idx in range(50):
    prompt = test_df.loc[idx, PROMPT_COL]
    p_original = get_probs(deberta_model, deberta_tok, prompt)
    p_augmented = get_probs(deberta_model, deberta_tok, INSTRUCTION_PREFIX + prompt)
    p_tta = (p_original + p_augmented) / 2

    top1_before = top1_label(p_original)
    top1_after = top1_label(p_tta)
    if top1_before != top1_after:
        tta_diff_count += 1

q6_answer = tta_diff_count
print(f"Q6: {q6_answer}")

# =========================================================
# Q7, Q8, Q9 — first 100 rows: DeBERTa vs Weighted Ensemble
# =========================================================
top1_diff_count = 0
positive_gain_count = 0
top3_diff_count = 0

for idx in range(100):
    prompt = test_df.loc[idx, PROMPT_COL]
    p_d = get_probs(deberta_model, deberta_tok, prompt)
    p_r = get_probs(roberta_model, roberta_tok, prompt)
    p_w = 0.7 * p_d + 0.3 * p_r

    # Q7 — Top-1 comparison
    if top1_label(p_d) != top1_label(p_w):
        top1_diff_count += 1

    # Q8 — Confidence gain
    conf_deberta = float(np.max(p_d))
    conf_ensemble = float(np.max(p_w))
    if (conf_ensemble - conf_deberta) > 0:
        positive_gain_count += 1

    # Q9 — Top-3 ordering comparison
    top3_d = top3_string(p_d)
    top3_w = top3_string(p_w)
    if top3_d != top3_w:
        top3_diff_count += 1

q7_answer = top1_diff_count
q8_answer = positive_gain_count
q9_answer = top3_diff_count

print(f"Q7: {q7_answer}")
print(f"Q8: {q8_answer}")
print(f"Q9: {q9_answer}")

# =========================================================
# Q10 — MAP@3 on first 100 validation samples (weighted ensemble)
# =========================================================
# TODO: adjust to your actual ground-truth column name in the validation set
GROUND_TRUTH_COL = "answer"  # e.g. "A", "B", "C", "D", "E"

val_df = pd.read_csv("test.csv")  # TODO: replace with your actual validation file if different

def apk(actual, predicted, k=3):
    if not actual:
        return 0.0
    if actual not in predicted[:k]:
        return 0.0
    return 1.0 / (predicted[:k].index(actual) + 1)

ap_scores = []
for idx in range(100):
    prompt = val_df.loc[idx, PROMPT_COL]
    actual = val_df.loc[idx, GROUND_TRUTH_COL]

    p_d = get_probs(deberta_model, deberta_tok, prompt)
    p_r = get_probs(roberta_model, roberta_tok, prompt)
    p_w = 0.7 * p_d + 0.3 * p_r

    order = np.argsort(-p_w)
    predicted_labels = [LABELS[i] for i in order]

    ap_scores.append(apk(actual, predicted_labels, k=3))

q10_answer = round(float(np.mean(ap_scores)), 4)
print(f"Q10: {q10_answer}")

# =========================================================
# Summary
# =========================================================
print("\n--- SUMMARY ---")
print(f"Q1: {q1_label}, probability of {q1_label} = {q1_prob}")
print(f"Q2: {q2_label}")
print(f"Q3: {q3_label}")
print(f"Q4: {q4_top3}")
print(f"Q5: {q5_answer}")
print(f"Q6: {q6_answer}")
print(f"Q7: {q7_answer}")
print(f"Q8: {q8_answer}")
print(f"Q9: {q9_answer}")
print(f"Q10: {q10_answer}")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'path/to/finetuned-deberta-v3-small'. Use `repo_type` argument if needed.